# Tasks 6 & 7: Feature Selection and Predictive Model

This notebook defines the features for predicting total piece travel time, trains an XGBoost Regressor, and evaluates its performance.

### The prediction problem

**Goal**: Predict the total travel time to the quench bath (`lifetime_bath_s`) using only data available **early in the process** — before the piece has finished the line.

**Why this is useful**: If we can predict the total time after only the 2nd strike, we can raise real-time alerts for pieces that are likely to be slow, allowing operators to investigate the cause while the piece is still on the line.

### Feature selection rationale

**Constraint**: the model must predict using only information available **after the 2nd strike** — approximately 18 seconds into the ~58-second journey. Any feature that requires waiting for later stages cannot be used, because by the time those values exist, the prediction is no longer useful.

#### Selected features

| Feature | Type | Available at | Why include it |
|---|---|---|---|
| `die_matrix` | Categorical (int) | Before processing | Each die has different tooling geometry and expected times — matrix 4974 has a median bath time of ~56s while 5091 has ~59s |
| `lifetime_2nd_strike_s` | Continuous (seconds) | After 2nd strike (~18s) | The earliest cumulative time — if the piece is already slow here, it will likely carry that delay through to the bath |
| `oee_cycle_time_s` | Continuous (seconds) | Rolling metric | Production rate context — slower OEE may correlate with systematic delays (robot trajectory adjustments, hydraulic pressure drops) |

#### Excluded features

| Feature | Why excluded |
|---|---|
| `lifetime_3rd_strike_s` | Available too late (~25s) — the piece is already halfway through the main press |
| `lifetime_4th_strike_s` | Available too late (~38s) — and has ~16% missing data from a sensor offline period |
| `lifetime_auxiliary_press_s` | Available too late (~55s) — only ~2s before the bath, prediction would be useless |
| `lifetime_general_s` | Equivalent to `lifetime_bath_s` — redundant with the target |
| `partial_*` columns | Derived from cumulative times that include late-stage data |
| `piece_id` | Not predictive — just an identifier |

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

GOLD_PATH = Path("../data/gold/pieces.parquet")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Fixed seed for reproducibility — same value used by the
# spec so that train/test splits and stochastic XGBoost behaviors
# (subsample/colsample) match
RANDOM_SEED = 42

## 1. Load gold dataset

In [3]:
df = pd.read_parquet(GOLD_PATH)

print(f"Loaded {len(df):,} pieces from gold")
print(f"Date range: {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
print(f"Die matrices: {sorted(df['die_matrix'].unique())}")
df.head()

Loaded 167,300 pieces from gold
Date range: 2025-11-06 → 2026-03-11
Die matrices: [np.int64(4974), np.int64(5052), np.int64(5090), np.int64(5091)]


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,oee_cycle_time_s,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s,after_gap,production_run_id
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,NaN,17.900000,6.700001,13.400000,16.599998,1.600002,True,0
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,NaN,17.900000,6.700001,13.300001,16.899998,1.600002,False,0
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,NaN,18.200001,6.599998,13.500000,17.000000,1.600002,False,0
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998,NaN,18.400000,6.700001,13.300001,17.099998,1.599998,False,0
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001,NaN,18.200001,6.599998,13.400002,17.299999,1.700001,False,0


## 2. Prepare features and target

- **Features (X)**: `die_matrix`, `lifetime_2nd_strike_s`, `oee_cycle_time_s`
- **Target (y)**: `lifetime_bath_s`

Drop rows where any feature or the target is NULL (missing drill data does not affect us here since we don't use drill as a feature).

In [5]:
FEATURES = ["die_matrix", "lifetime_2nd_strike_s", "oee_cycle_time_s"]
TARGET = "lifetime_bath_s"

# Drop rows where any feature or the target is NaN. Why drop, not impute:
# - lifetime_2nd_strike_s: ~0 missing in clean data
# - oee_cycle_time_s: ~23% missing (NULL when out of the 11-16s range, by design
#   of the cleaning rule). With ~167k pieces, dropping leaves enough data for
#   training. The inference service (Task 7) handles missing OEE at prediction
#   time by substituting the median.
# - lifetime_bath_s (target): cannot train on rows without a target value
data = df[FEATURES + [TARGET]].dropna()

X = data[FEATURES]
y = data[TARGET]

print(f"Training data: {len(data):,} pieces ({len(data)/len(df)*100:.1f}% of gold)")
print(f"Features: {FEATURES}")
print(f"Target:   {TARGET}")
print(f"\nFeature statistics:")
X.drop(columns=["die_matrix"]).describe().round(2)

Training data: 125,941 pieces (75.3% of gold)
Features: ['die_matrix', 'lifetime_2nd_strike_s', 'oee_cycle_time_s']
Target:   lifetime_bath_s

Feature statistics:


,lifetime_2nd_strike_s,oee_cycle_time_s
count,125941.00,125941.00
mean,18.58,13.89
std,2.18,0.58
min,9.80,11.00
25%,16.90,13.47
50%,18.00,13.81
75%,19.80,14.21
max,31.00,16.00


## 3. Feature correlation with target

How strongly does each feature correlate with the total bath time? High correlation suggests predictive value.

In [6]:
# Pearson correlation between each feature and the target.
# Note: die_matrix is encoded as int, so its linear correlation appears weak —
# matrices have different baselines but the relationship is non-linear, which
# tree-based models like XGBoost capture naturally through splits.
correlations = data[FEATURES].apply(lambda col: col.corr(y))

print("Pearson correlation with lifetime_bath_s:")
for feat, corr in correlations.items():
    print(f"  {feat:<25s}  {corr:+.3f}")

Pearson correlation with lifetime_bath_s:
  die_matrix                 +0.240
  lifetime_2nd_strike_s      +0.815
  oee_cycle_time_s           +0.443


## 4. Train/test split

Split 80/20 with a fixed random seed for reproducibility. Stratify by die_matrix to ensure each matrix is represented proportionally in both sets.

In [7]:
# 80/20 split, stratified by die_matrix.
# Why stratify: smaller matrices (e.g. 4974 with ~16k pieces) could be
# under-represented in the test set with a random split, making the
# per-matrix evaluation in section 7 unreliable. Stratification guarantees
# each matrix appears in train and test in proportion to its size.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=X["die_matrix"],
)

print(f"Train set: {len(X_train):,} pieces")
print(f"Test set:  {len(X_test):,} pieces")
print(f"\nTrain matrix distribution:")
print(X_train["die_matrix"].value_counts().sort_index().to_string())
print(f"\nTest matrix distribution:")
print(X_test["die_matrix"].value_counts().sort_index().to_string())

Train set: 100,752 pieces
Test set:  25,189 pieces

Train matrix distribution:
die_matrix
4974    10713
5052    11871
5090    48676
5091    29492

Test matrix distribution:
die_matrix
4974     2678
5052     2968
5090    12170
5091     7373


## 5. Train XGBoost Regressor

XGBoost is chosen because:
- Handles mixed feature types (categorical die_matrix + continuous times)
- Robust to the remaining noise in the data
- Fast training on ~100k rows
- Produces feature importance rankings

In [9]:
# Hyperparameters per the model design doc (docs/05_PredictiveModel.md).
# Each value is deliberately chosen:
# - n_estimators=200 + learning_rate=0.1: standard 200-tree configuration —
#   enough trees to converge with a moderate learning rate, without overfitting
# - max_depth=6: deep enough to capture die_matrix × lifetime interactions
#   (e.g., "for matrix 5091, a slow 2nd strike has more impact than for 4974")
# - subsample=0.8: each tree trains on 80% of rows (random) — reduces overfitting
# - colsample_bytree=0.8: each tree sees 80% of features — adds further regularization
model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

model.fit(X_train, y_train)
print("Training complete")

Training complete


## 6. Evaluate on test set

Key metrics:
- **RMSE**: root mean squared error (same unit as target — seconds)
- **MAE**: mean absolute error (average prediction error in seconds)
- **R²**: coefficient of determination (1.0 = perfect, 0.0 = no better than mean)

In [10]:
y_pred = model.predict(X_test)

# Three complementary metrics:
# - RMSE: penalizes large errors more (squared) — sensitive to outliers
# - MAE: average absolute error in seconds — most intuitive for users
# - R²: fraction of variance explained — model fit quality (1.0 = perfect)
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
mae = float(mean_absolute_error(y_test, y_pred))
r2 = float(r2_score(y_test, y_pred))

print(f"Test set metrics:")
print(f"  RMSE: {rmse:.3f} s")
print(f"  MAE:  {mae:.3f} s")
print(f"  R²:   {r2:.4f}")
print()
print(f"Interpretation: on a typical bath time of ~58s,")
print(f"  MAE = {mae:.2f}s represents a ~{mae/58*100:.1f}% relative error.")
print(f"  Sufficient for real-time alerting where the goal is to detect")
print(f"  multi-second delays, not sub-second precision.")

Test set metrics:
  RMSE: 1.363 s
  MAE:  0.662 s
  R²:   0.7775

Interpretation: on a typical bath time of ~58s,
  MAE = 0.66s represents a ~1.1% relative error.
  Sufficient for real-time alerting where the goal is to detect
  multi-second delays, not sub-second precision.


## 7. Performance per die matrix

Check if the model performs equally well across all matrices, or if some are harder to predict.

In [11]:
# Per-matrix evaluation: does the model work equally well for every matrix,
# or is one harder to predict? If a matrix has significantly worse metrics
# than the others, the model is biased and shouldn't be trusted for that
# matrix in production — operators would get unreliable alerts.
test_results = X_test.copy()
test_results["actual"] = y_test
test_results["predicted"] = y_pred

per_matrix = (
    test_results.groupby("die_matrix")
    .apply(
        lambda g: pd.Series({
            "n_pieces": len(g),
            "rmse": float(np.sqrt(mean_squared_error(g["actual"], g["predicted"]))),
            "mae": float(mean_absolute_error(g["actual"], g["predicted"])),
            "r2": float(r2_score(g["actual"], g["predicted"])),
        }),
        include_groups=False,
    )
    .round(3)
)
print("Per-matrix test set performance:")
per_matrix

Per-matrix test set performance:


,n_pieces,rmse,mae,r2
die_matrix,,,,
4974,2678.0,1.079,0.503,0.704
5052,2968.0,1.081,0.475,0.816
5090,12170.0,1.608,0.845,0.695
5091,7373.0,1.089,0.494,0.860


## 8. Feature importance

Which features contribute most to the prediction? This validates the feature selection rationale.

In [12]:
# XGBoost gain-based feature importance: how much each feature contributes
# to reducing the loss across all splits in all trees.
# Expected: lifetime_2nd_strike_s should dominate by a wide margin —
# this confirms that early-stage timing is the strongest predictor of
# total travel time, validating the feature selection rationale.
importance = pd.Series(
    model.feature_importances_,
    index=FEATURES,
).sort_values(ascending=False)

print("Feature importance (gain-based, normalized):")
for feat, imp in importance.items():
    bar = "█" * int(imp * 50)
    print(f"  {feat:<25s} {imp:.4f}  {bar}")

Feature importance (gain-based, normalized):
  lifetime_2nd_strike_s     0.6807  ██████████████████████████████████
  oee_cycle_time_s          0.2552  ████████████
  die_matrix                0.0641  ███


## 9. Save model and metadata

Save the trained model and its metadata for use by the inference service (Task 8).

In [ ]:
# Save the trained model in XGBoost's native JSON format. Why JSON:
# - portable across XGBoost versions
# - loadable without retraining (`xgb.XGBRegressor().load_model(...)`)
# - compatible with the SageMaker built-in XGBoost container used in Task 10
model_path = MODELS_DIR / "xgboost_bath_predictor.json"
model.save_model(model_path)
print(f"Model saved to {model_path}")

# Save metadata: features, target, hyperparameters, metrics, importances.
# The Predictor class (Task 7) loads this to:
# - know which columns to use as features
# - validate die_matrix values against the trained set
# - report model metrics alongside predictions
# - impute missing OEE with the median (computed once here, persisted)
metadata = {
    "model_type": "xgboost.XGBRegressor",
    "features": FEATURES,
    "target": TARGET,
    "hyperparameters": {
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": RANDOM_SEED,
    },
    "train_size": len(X_train),
    "test_size": len(X_test),
    "metrics": {
        "rmse": round(rmse, 4),
        "mae": round(mae, 4),
        "r2": round(r2, 4),
    },
    "feature_importance": {feat: float(imp) for feat, imp in importance.items()},
    "die_matrices": sorted(int(m) for m in df["die_matrix"].unique()),
    "oee_median_for_imputation": float(df["oee_cycle_time_s"].median()),
}
metadata_path = MODELS_DIR / "model_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved to {metadata_path}")

## 10. Summary

In [ ]:
print("=" * 60)
print("MODEL TRAINING SUMMARY")
print("=" * 60)
print(f"\nFeatures used:    {', '.join(FEATURES)}")
print(f"Target:           {TARGET}")
print(f"Training pieces:  {len(X_train):,}")
print(f"Test pieces:      {len(X_test):,}")

print(f"\nTest set metrics:")
print(f"  RMSE: {rmse:.3f} s")
print(f"  MAE:  {mae:.3f} s")
print(f"  R²:   {r2:.4f}")

print(f"\nFeature importance ranking:")
for i, (feat, imp) in enumerate(importance.items(), 1):
    print(f"  {i}. {feat}: {imp:.4f}")

print(f"\nArtifacts saved:")
print(f"  {model_path}")
print(f"  {metadata_path}")
print("\n" + "=" * 60)